# Code : Courbes avec torsion

In [1]:
def EllipticCurve_with_order(m, *, D=None):
    r"""
    Return an iterator for elliptic curves over finite fields with the given order. The curves are
    computed using the Complex Multiplication (CM) method.

    A :class:`~sage.structure.factorization.Factorization` can be passed for ``m``, in which case
    the algorithm is more efficient.

    If ``D`` is specified, it is used as the discriminant.

    EXAMPLES::

        sage: from sage.schemes.elliptic_curves.ell_finite_field import EllipticCurve_with_order
        sage: E = next(EllipticCurve_with_order(1234)); E  # random
        Elliptic Curve defined by y^2 = x^3 + 1142*x + 1209 over Finite Field of size 1237
        sage: E.order() == 1234
        True

    When ``iter`` is set, the function returns an iterator of all elliptic curves with the given
    order::

        sage: from sage.schemes.elliptic_curves.ell_finite_field import EllipticCurve_with_order
        sage: it = EllipticCurve_with_order(21); it
        <generator object EllipticCurve_with_order at 0x...>
        sage: E = next(it); E  # random
        Elliptic Curve defined by y^2 = x^3 + 6*x + 14 over Finite Field of size 23
        sage: E.order() == 21
        True
        sage: Es = [E] + list(it); Es  # random
        [Elliptic Curve defined by y^2 = x^3 + 6*x + 14 over Finite Field of size 23,
         Elliptic Curve defined by y^2 = x^3 + 12*x + 4 over Finite Field of size 23,
         Elliptic Curve defined by y^2 = x^3 + 5*x + 2 over Finite Field of size 23,
         Elliptic Curve defined by y^2 = x^3 + (z2+3) over Finite Field in z2 of size 5^2,
         Elliptic Curve defined by y^2 = x^3 + (2*z2+2) over Finite Field in z2 of size 5^2,
         Elliptic Curve defined by y^2 = x^3 + 7*x + 1 over Finite Field of size 19,
         Elliptic Curve defined by y^2 = x^3 + 17*x + 10 over Finite Field of size 19,
         Elliptic Curve defined by y^2 = x^3 + 5*x + 12 over Finite Field of size 17,
         Elliptic Curve defined by y^2 = x^3 + 9*x + 1 over Finite Field of size 17,
         Elliptic Curve defined by y^2 = x^3 + 7*x + 6 over Finite Field of size 17,
         Elliptic Curve defined by y^2 = x^3 + z3^2*x^2 + (2*z3^2+z3) over Finite Field in z3 of size 3^3,
         Elliptic Curve defined by y^2 = x^3 + (z3^2+2*z3+1)*x^2 + (2*z3^2+2*z3) over Finite Field in z3 of size 3^3,
         Elliptic Curve defined by y^2 = x^3 + (z3^2+z3+1)*x^2 + (2*z3^2+1) over Finite Field in z3 of size 3^3,
         Elliptic Curve defined by y^2 + (z4^2+z4+1)*y = x^3 over Finite Field in z4 of size 2^4,
         Elliptic Curve defined by y^2 + (z4^2+z4)*y = x^3 over Finite Field in z4 of size 2^4,
         Elliptic Curve defined by y^2 = x^3 + 18*x + 26 over Finite Field of size 29,
         Elliptic Curve defined by y^2 = x^3 + 11*x + 19 over Finite Field of size 29,
         Elliptic Curve defined by y^2 = x^3 + 4 over Finite Field of size 19,
         Elliptic Curve defined by y^2 = x^3 + 19 over Finite Field of size 31,
         Elliptic Curve defined by y^2 = x^3 + 4 over Finite Field of size 13]
        sage: all(E.order() == 21 for E in Es)
        True

    Indeed, we can verify that this is correct. Hasse's bounds tell us that
    `p \leq 50` (approximately), and the rest can be checked via bruteforce::

        sage: for p in prime_range(50):
        ....:     for j in range(p):
        ....:         E0 = EllipticCurve(GF(p), j=j)
        ....:         for Et in E0.twists():
        ....:             if Et.order() == 21:
        ....:                 assert any(Et.is_isomorphic(E) for E in Es)

    .. NOTE::

        The output curves are not deterministic, as :func:`EllipticCurve_finite_field.twists` is not
        deterministic. However, the order of the j-invariants and base fields is fixed.

    AUTHORS:

     - Gareth Ma and Giacomo Pope (Sage Days 123): initial version
    """
    from sage.arith.misc import is_prime_power, factor
    from sage.quadratic_forms.binary_qf import BinaryQF
    from sage.structure.factorization import Factorization
    from sage.schemes.elliptic_curves.cm import hilbert_class_polynomial

    def find_q(m, m4_fac, D):
        for t, _ in BinaryQF(1, 0, -D).solve_integer(m4_fac, _flag=3):
            yield m + 1 - t
            yield m + 1 + t

    if isinstance(m, Factorization):
        m4_fac = m * factor(4)
        m_val = m.value()
    else:
        m4_fac = factor(m * 4)
        m_val = m

    if D is None:
        Ds = (D for D in range(-1, -4 * m_val - 1, -1) if D % 4 in [0, 1])
    else:
        assert D < 0 and D % 4 in [0, 1]
        Ds = [D]

    seen = set()
    for D in Ds:
        for q in find_q(m_val, m4_fac, D):
            if not is_prime_power(q):
                continue

            H = hilbert_class_polynomial(D)
            for j0 in H.roots(ring=GF(q), multiplicities=False):
                E = EllipticCurve(j=j0)
                for Et in E.twists():
                    if any(Et.is_isomorphic(E) for E in seen):
                        continue
                    # This tests whether the curve has given order
                    if Et.has_order(m_val):
                        # TODO: remove after 38617
                        Et.set_order(m_val, check=False)
                        seen.add(Et)
                        yield Et,D

#it = EllipticCurve_with_order((2^32).factor())

#2^190 : torsion 2^12, définie sur p. disc = -7
#2^32 : 1er : torsion 2^16, définie sur p^2, disc = -3 
#      2ieme : torsion 2^11 définie sur p, disc = -7
#      Autres courbes interessantes
#      Probleme : h(-7) = 1. Solution à partir de -39

In [2]:

def EllipticCurve_with_Torsion(ell,size,tmin,nb_class):

    # On cherche un exemple de courbe ordinaire où la ell^n-torsion est définie pour un certain n < size

    #On préfère un discriminant dont le nombre de classe n'est pas 1, pour travailler avec des idéaux non principaux

    card = ell^size
    it = EllipticCurve_with_order((card).factor())  #On choisit comme cardinal une puissance de deux, ici arbitraire
    Tmin = ell^tmin
    assert card%Tmin == 0
    nK = 1
    T = 1
    
    while nK < nb_class or T < Tmin:

        E,D = next(it)
        A = E.abelian_group()
        inv = A.invariants()
        assert inv[1] % inv[0] == 0
        T = inv[0]
        K.<rK> = QuadraticField(D)
        dK = K.discriminant()
        K.<rK> = QuadraticField(dK)
        O = K.maximal_order()
        f = O.conductor()
        nK = K.class_number()

    return E,D,f,T
   

EllipticCurve_with_Torsion(2,32,14,4)

(Elliptic Curve defined by y^2 = x^3 + 2857265638*x + 3935716393 over Finite Field of size 4295049217,
 -39,
 1,
 16384)

# Code : Génération de diamants

In [26]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
import time


def Ideal_of_norm(l,D,f):

    #Trouver un idéal de norme l premier.
    #On se place dans un ordre de discriminant D et de conducteur f.

    K.<rK> = QuadraticField(D)
    dK = K.discriminant()
    K.<rK> = QuadraticField(dK)
    wK = (dK + rK)/2
    O = K.order([1,f*wK])
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) 
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"


def ideal_to_element(a,L,O):
    
    #On suppose a equivalent à L. On cherche alpha dans L tel que a = ( (alpha)bar / N(L))L.
    
    assert a.is_equivalent(L)
    B = a*(L.conjugate())
    alphabar = (B.gens_reduced())[0]
    alpha = alphabar.conjugate()
    assert NumberFieldOrderIdeal(O,alphabar) == B
    assert alpha in L
    return alpha

def element_to_ideal(alpha,L,O):
    
    #On suppose alpha dans L et on calcule l'idéal équivalent associé a = ( (alpha)bar / N(L))L.
    
    assert alpha in L
    alphabar = alpha.conjugate()
    B = NumberFieldOrderIdeal(O,alphabar)
    a2 = B*L
    if a2.is_principal():
        g = a2.gens_reduced()
        a = NumberFieldOrderIdeal(O,g[0]/(L.norm()))
    else:
        g1, g2 = a2.gens_reduced()
        a = NumberFieldOrderIdeal(O,[g1/(L.norm()), g2/(L.norm())])
    assert a.is_equivalent(L)
    return a

def make_liste_ideq(L,O,m,CE):   

    # On établit une liste d'idéaux équivalent à L, de normes minimales. 
    # paramètre m : Le nombre d'idéaux que l'on teste 2m^2 + m. 
    # On ne retient que les idéaux de normes premier avec CE.

    qL = L.quadratic_form()
    rL = qL.reduced_form()
    RL = NumberFieldOrderIdeal(O,rL)
    
    rK = O.number_field().gens()[0]
    f = O.conductor()
    alpha = rL[0]
    beta = (-rL[1] + f*rK)/2   #N'utilise pas LLL, contrairement à l'implémentation sage de Pegasis.
    
    liste_ideq = []
    
    for x in [0 .. m]:   
        for y in [-m .. m]:
            if (x != 0 or y > 0) and x.gcd(y) == 1:   #Pour ne pas calculer I et kI avec k entier.
                gamma = x*alpha + y*beta    #Pour ne pas creer gamma et -gamma, on evite les cas x < 0, ou (x = 0 et y <= 0), 
                
                if gamma == 0:
                    raise ValueError('erreur base courte liée') #n'est pas censé arriver. 
                    
                I = element_to_ideal(gamma,RL,O)
                NI = I.norm()
                if NI.gcd(CE) != 1:
                    continue
                    
                #liste_ideq.append([I,Nk,Ne,Ne_facto]) trier selon Nk ?
                i = len(liste_ideq)
                if NI == 1:
                    continue
                if i == 0:
                    liste_ideq.append([I,NI])
                else:
                    i = i-1
                    test = NI < liste_ideq[i][1]
                    while i > -1 and test :
                        i = i - 1
                        test = NI < liste_ideq[i][1]
                    liste_ideq.insert(i+1,[I,NI])

    return liste_ideq

def Ideal_random(D,f):
    
    l = next_prime(randint(10^20, 10^21))  #Pour générer un nombre premier aléatoire (une classe aléatoire)
    while kronecker(D,l) != 1:
        l = next_prime(l)

    L = Ideal_of_norm(l,D,f)
    return L

    
def T_diamant(E,D,f,Tmin,Tmax,L):
    
    #Paramètre par défaut : m_id pour le nombre d'idéaux testés
    p = E.base_ring().characteristic()
    m_id = isqrt(p.nbits())+1
    
    #Retourne un diamant de taille divisant T, sans idéaux friables ni endomorphismes
    K.<rK> = QuadraticField(D)
    dK = K.discriminant()
    K.<rK> = QuadraticField(dK)
    wK = (dK + rK)/2
    O = K.order([1,f*wK])
    
    assert O.conductor() == f and O.discriminant() == D and D == (f^2)*dK
    
    CE = E.cardinality_pari()
    
    liste_ideq = make_liste_ideq(L,O,m_id,CE)
    
    k_id = len(liste_ideq)
    sols = []
    sols_ext = []
    tentatives = 0

    #Parcourir les couples d'idéaux minimsant Nk1Nk2 - Nk1 - Nk2

    for i in [0 .. k_id-2]:
        b_prep = liste_ideq[i]
        b, N1 = b_prep
        
        for j in [i+1 .. k_id-1]:
            c_prep = liste_ideq[j]
            c, N2 = c_prep
            
            tentatives = tentatives + 1
            
            if N1.gcd(N2) != 1 or (N1 + N2) < Tmin:
                continue
                
            if Tmax % (N1 + N2) == 0:
                print('diamant:',N1,'+',N2,'=',N1 + N2)
                print('tentatives :', tentatives)
                print('indices idéaux :', i, j)
                return [N1,N2,N1 + N2,b,c]

    raise ValueError("Aucune solution trouvée")

In [27]:
E,D,f,T = EllipticCurve_with_Torsion(2,32,14,4)

l = 116617643378184942397 #Pour garder le même exemple

L = Ideal_of_norm(l,D,f)
print(L.quadratic_form().reduced_form())
T_diamant(E,D,f,16,T,L)

x^2 + x*y + 10*y^2
diamant: 75 + 181 = 256
tentatives : 70
indices idéaux : 5 10


[75,
 181,
 256,
 Ideal (13/2*rK + 3/2, 25*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I,
 Ideal (109/2*rK + 1/2, 181*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I]

In [28]:
E,D,f,T = EllipticCurve_with_Torsion(2,32,14,4)
L = Ideal_random(D,f)
print(L.quadratic_form().reduced_form())
T_diamant(E,D,f,32,T,L)

2*x^2 + x*y + 5*y^2
diamant: 5 + 59 = 64
tentatives : 6
indices idéaux : 0 6


[5,
 59,
 64,
 Ideal (1/2*rK + 1/2, 5*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I,
 Ideal (11/2*rK + 1/2, 59*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 39 with rK = 6.244997998398398?*I]

In [29]:
E,D,f,T = EllipticCurve_with_Torsion(2,34,8,3)

In [32]:
L = Ideal_random(D,f)
print(L.quadratic_form().reduced_form())
T_diamant(E,D,f,32,T,L)

x^2 + x*y + 6*y^2
diamant: 27 + 101 = 128
tentatives : 22
indices idéaux : 1 8


[27,
 101,
 128,
 Ideal (41/2*rK + 1/2, 27*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 23 with rK = 4.795831523312720?*I,
 Ideal (169/2*rK + 1/2, 101*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 23 with rK = 4.795831523312720?*I]